# VoiceHub inference: TTS, ASR, and VAD

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/inference.ipynb)

This notebook demonstrates VoiceHub's task-specific inference factories, lazy model loading, normalized outputs, and a VAD-to-ASR pipeline. Registry cells are offline-safe. Checkpoint downloads and audio execution remain disabled until you opt in.

Use only checkpoints and voice references whose licenses and permissions fit your application.

## 0. Install VoiceHub

The default package contains the runtime dependencies for every built-in TTS, ASR, and VAD integration. The setup cell installs VoiceHub only when it is not already importable. Pin a release tag or commit SHA for a recorded experiment.

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_VOICEHUB = importlib.util.find_spec("voicehub") is None
VOICEHUB_REVISION = "main"  # Prefer a release tag or full commit SHA.

if INSTALL_VOICEHUB:
    package = (
        "voicehub @ git+https://github.com/kadirnar/voicehub.git@"
        f"{VOICEHUB_REVISION}"
    )
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        package,
    ])


## 1. Configure explicit opt-in execution

Keep the three execution flags disabled while inspecting the catalog. Set only the task you want to run to `True` after selecting a checkpoint and device.

In [ ]:
from pathlib import Path

from voicehub import SpeechTask, __version__, list_model_specs

RUN_TTS = False
RUN_ASR = False
RUN_VAD = False

DEVICE = "cpu"  # Use "cuda" on a compatible accelerator.
VAD_DEVICE = "cpu"  # Native WebRTC VAD is intentionally CPU-only.
AUDIO_PATH = Path("data/example.wav")
ARTIFACTS_DIR = Path("artifacts/inference")

TTS_MODEL_TYPE = "kokoro"
TTS_CHECKPOINT = "hexgrad/Kokoro-82M"
ASR_MODEL_TYPE = "asr_moonshine"
ASR_CHECKPOINT = "UsefulSensors/moonshine-tiny"
VAD_MODEL_TYPE = "vad_webrtc"
VAD_CHECKPOINT = "webrtc-vad"

catalog = {
    spec.model_type: spec
    for spec in list_model_specs(task=None)
}
print("VoiceHub", __version__, "registered", len(catalog), "integrations")


## 2. Discover models without loading weights

Registry inspection does not import a model runtime or download a checkpoint. Review the architecture, default checkpoint, capabilities, and training boundary before construction.

In [ ]:
task_counts = {}
for task in SpeechTask:
    matching = sorted(
        spec.model_type
        for spec in catalog.values()
        if spec.task is task
    )
    task_counts[task.value] = len(matching)
    print(f"{task.value}: {len(matching)}")
    print("  ", ", ".join(matching))

for selected in (TTS_MODEL_TYPE, ASR_MODEL_TYPE, VAD_MODEL_TYPE):
    spec = catalog[selected]
    print({
        "model_type": spec.model_type,
        "task": spec.task.value,
        "architecture": spec.architecture,
        "default_checkpoint": spec.default_model_path,
        "capabilities": tuple(spec.capabilities),
        "training": spec.training.support.value,
    })


## 3. Configure requests and inspect normalized outputs

Request and output types can be created without loading a checkpoint. This makes configuration validation and downstream application code testable independently from model execution.

In [ ]:
from voicehub import (
    ASRInferenceConfig,
    ASROutput,
    ASRSegment,
    SpeechSegment,
    TTSGenerationConfig,
    TTSOutput,
    VADInferenceConfig,
    VADOutput,
)

tts_request = TTSGenerationConfig(
    seed=42,
    speed=1.0,
    output_file=ARTIFACTS_DIR / "tts.wav",
)
asr_request = ASRInferenceConfig(
    language="en",
    task="transcribe",
    num_beams=1,
    max_new_tokens=128,
)
vad_request = VADInferenceConfig(
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=100,
    speech_pad_ms=30,
)

tts_preview = TTSOutput(
    audio=[0.0, 0.1, -0.1],
    sample_rate=24_000,
    metadata={"backend": "preview"},
)
asr_preview = ASROutput(
    text="VoiceHub normalizes speech outputs.",
    segments=(
        ASRSegment(
            text="VoiceHub normalizes speech outputs.",
            start=0.0,
            end=1.2,
            language="en",
        ),
    ),
    language="en",
    duration=1.2,
)
vad_preview = VADOutput(
    segments=(SpeechSegment(start=0.2, end=0.7, score=0.9),),
    duration=1.0,
    sample_rate=16_000,
)

print(
    tts_preview.sample_rate,
    asr_preview.text,
    vad_preview.speech_duration,
    vad_preview.contains(0.25),
)


## 4. Text-to-speech inference

Construction is lazy. The first `generate()` call loads the checkpoint. `TTSOutput` normalizes waveform samples, sample rate, output path, and model metadata.

In [ ]:
tts_output = None

if RUN_TTS:
    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    tts_model = AutoModelForTextToSpeech.from_pretrained(
        TTS_CHECKPOINT,
        model_type=TTS_MODEL_TYPE,
        device=DEVICE,
        torch_dtype="auto",
        lazy_load=True,
    )
    tts_output = tts_model.generate(
        "Hello",
        generation_config=TTSGenerationConfig(
            seed=42,
            speed=1.0,
            output_file=ARTIFACTS_DIR / "tts.wav",
        ),
        voice="af_heart",
        phonemes="həlˈoʊ",
    )
    print(tts_output.sample_rate, tts_output.file_path)
    try:
        from IPython.display import Audio, display
    except ModuleNotFoundError:
        pass
    else:
        display(Audio(tts_output.audio, rate=tts_output.sample_rate))
else:
    print("TTS is disabled. Set RUN_TTS=True after reviewing the checkpoint.")


## 5. Speech-recognition inference

ASR accepts a path, array, tensor, mapping, or `AudioInput`. The normalized result exposes text, segments, words when available, language, and provider metadata.

In [ ]:
asr_output = None

if RUN_ASR:
    from voicehub import AutoModelForSpeechRecognition

    if not AUDIO_PATH.is_file():
        raise FileNotFoundError(AUDIO_PATH)
    asr_model = AutoModelForSpeechRecognition.from_pretrained(
        ASR_CHECKPOINT,
        model_type=ASR_MODEL_TYPE,
        device=VAD_DEVICE,
        torch_dtype="float32",
        lazy_load=True,
    )
    asr_output = asr_model.transcribe(
        AUDIO_PATH,
        inference_config=asr_request,
    )
    print(asr_output.text)
    for segment in asr_output.segments:
        print(segment.start, segment.end, segment.text)
else:
    print("ASR is disabled. Set RUN_ASR=True and provide AUDIO_PATH.")


## 6. Voice-activity detection

VAD returns ordered, non-overlapping speech regions. Threshold and duration controls remain request configuration rather than hidden global state.

In [ ]:
vad_output = None

if RUN_VAD:
    from voicehub import (
        AutoModelForVoiceActivityDetection,
        VADInferenceConfig,
    )

    if not AUDIO_PATH.is_file():
        raise FileNotFoundError(AUDIO_PATH)
    vad_model = AutoModelForVoiceActivityDetection.from_pretrained(
        VAD_CHECKPOINT,
        model_type=VAD_MODEL_TYPE,
        device=DEVICE,
        sample_rate=16_000,
        aggressiveness=2,
        frame_duration_ms=30,
    )
    vad_output = vad_model.detect(
        AUDIO_PATH,
        inference_config=VADInferenceConfig(
            threshold=0.5,
            min_speech_duration_ms=250,
            min_silence_duration_ms=150,
            speech_pad_ms=30,
            return_frames=False,
        ),
    )
    for segment in vad_output.segments:
        print(f"{segment.start:.3f}s -> {segment.end:.3f}s")
else:
    print("VAD is disabled. Set RUN_VAD=True and provide AUDIO_PATH.")


## 7. Compose VAD and ASR explicitly

Keep detection and recognition as separate lifecycles so timebases and model state stay auditable. The example below loads normalized audio once, slices each detected region, and transcribes it while retaining absolute VAD timestamps.

In [ ]:
if RUN_VAD and RUN_ASR:
    from voicehub import load_audio

    normalized_audio = load_audio(AUDIO_PATH, target_sampling_rate=16_000)
    for region in vad_output.segments:
        start, end = region.sample_bounds(normalized_audio.sampling_rate)
        transcript = asr_model.transcribe(
            normalized_audio.waveform[start:end],
            sampling_rate=normalized_audio.sampling_rate,
            inference_config=asr_request,
        )
        print(region.start, region.end, transcript.text)
else:
    print("Enable both RUN_VAD and RUN_ASR to execute the composed pipeline.")


## Next steps

- Read the [inference guide](https://kadirnar.github.io/voicehub/guides/inference/) for model-specific conditioning and serving strategies.
- Use the [speech-recognition guide](https://kadirnar.github.io/voicehub/guides/speech-recognition/) and [VAD guide](https://kadirnar.github.io/voicehub/guides/voice-activity-detection/) for provider-specific controls.
- Continue with [data preparation](data_preparation.ipynb) and [training](training.ipynb) before adapting weights.